# Modeling

Ce notebook contient le code du module `src/modeling.py` pour référence.

**Note** : Pour l'entraînement, utilisez `house_price_02_essais.ipynb` qui contient l'exemple complet.

In [ ]:
# Imports
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from src.modeling import ModelTrainer

In [ ]:
## Utilisation

Le code complet de `ModelTrainer` est dans `src/modeling.py`.  
Voir `house_price_02_essais.ipynb` pour un exemple d'utilisation complet.
    """Class for training and evaluating models."""
    
    def __init__(self, experiment_name: str = "house_price_prediction"):
        """
        Initialize ModelTrainer.
        
        Args:
            experiment_name: Name of MLFlow experiment
        """
        self.experiment_name = experiment_name
        mlflow.set_experiment(experiment_name)
        self.models = {}
        self.best_model = None
        self.best_score = float('inf')
        
    def train_model(self, model: Any, X_train: pd.DataFrame, y_train: pd.Series,
                   X_val: Optional[pd.DataFrame] = None, 
                   y_val: Optional[pd.Series] = None,
                   model_name: str = "model",
                   params: Optional[Dict] = None,
                   use_mlflow: bool = True) -> Dict[str, float]:
        """
        Train a model and evaluate it.
        
        Args:
            model: Model object to train
            X_train: Training features
            y_train: Training target
            X_val: Validation features (optional)
            y_val: Validation target (optional)
            model_name: Name of the model
            params: Model parameters to log
            use_mlflow: Whether to log to MLFlow
            
        Returns:
            Dictionary of evaluation metrics
        """
        logger.info(f"Training {model_name}...")
        
        # Train model
        model.fit(X_train, y_train)
        
        # Predictions
        train_pred = model.predict(X_train)
        
        # Calculate metrics
        train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
        train_mae = mean_absolute_error(y_train, train_pred)
        train_r2 = r2_score(y_train, train_pred)
        
        metrics = {
            'train_rmse': train_rmse,
            'train_mae': train_mae,
            'train_r2': train_r2
        }
        
        # Validation metrics if provided
        if X_val is not None and y_val is not None:
            val_pred = model.predict(X_val)
            val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
            val_mae = mean_absolute_error(y_val, val_pred)
            val_r2 = r2_score(y_val, val_pred)
            
            metrics.update({
                'val_rmse': val_rmse,
                'val_mae': val_mae,
                'val_r2': val_r2
            })
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train, y_train, 
                                      cv=KFold(n_splits=5, shuffle=True, random_state=42),
                                      scoring='neg_mean_squared_error')
            cv_rmse = np.sqrt(-cv_scores.mean())
            metrics['cv_rmse'] = cv_rmse
            metrics['cv_std'] = np.sqrt(cv_scores.std())
        
        # Log to MLFlow
        if use_mlflow:
            with mlflow.start_run(run_name=model_name):
                if params:
                    mlflow.log_params(params)
                mlflow.log_metrics(metrics)
                mlflow.sklearn.log_model(model, "model")
        
        # Store model
        self.models[model_name] = {
            'model': model,
            'metrics': metrics
        }
        
        # Update best model
        if X_val is not None and y_val is not None:
            if val_rmse < self.best_score:
                self.best_score = val_rmse
                self.best_model = model_name
        
        logger.info(f"{model_name} - RMSE: {metrics.get('val_rmse', train_rmse):.2f}")
        
        return metrics
    
    def get_best_model(self) -> Optional[Any]:
        """Get the best performing model."""
        if self.best_model:
            return self.models[self.best_model]['model']
        return None
    
    def save_model(self, model: Any, filepath: str):
        """
        Save model to file.
        
        Args:
            model: Model to save
            filepath: Path to save model
        """
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, filepath)
        logger.info(f"Model saved to {filepath}")
    
    def load_model(self, filepath: str) -> Any:
        """
        Load model from file.
        
        Args:
            filepath: Path to model file
            
        Returns:
            Loaded model
        """
        model = joblib.load(filepath)
        logger.info(f"Model loaded from {filepath}")
        return model